[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/monacofj/misda/blob/main/benchmarks/optimization.ipynb)

# MISDA — optimization benchmark

This notebook tests whether replacing the full objective set of a multi-objective optimization problem by the subset selected by MISDA preserves optimization quality and changes convergence behavior.

The comparison is paired:

- **Full** optimizes all original objectives.
- **Reduced** optimizes only the objectives selected by MISDA.
- Both use NSGA-III with the same decision domain, population, generations, and run seed.
- Reduced decision vectors are re-evaluated on the **original M-objective problem** before every comparison.
- This original-space re-evaluation is benchmark instrumentation only; it never feeds back into the Reduced search.

The analytical Pareto front of the original MOP is the common ruler for both treatments.


In [ ]:
from pathlib import Path
import subprocess
import sys

# In a repository checkout, test local code. In Colab, install main.
repo_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()), None)
REMOTE_REF = "main"
target = f"{repo_root}[benchmarks]" if repo_root is not None else f"misda[benchmarks] @ git+https://github.com/monacofj/misda.git@{REMOTE_REF}"
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", target])


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

import misda
import moeabench as mb
from moeabench.core.run import Population

M = 10
MISDA_SAMPLE = 1000
GT_POINTS = 5000
POPULATION = 100
GENERATIONS = 300
MISDA_SEED = 123
MOEA_SEED = 321
HV_MC_SAMPLES = 100_000


## Experimental protocol

MISDA receives a reproducible pre-optimization screening sample from the original decision domain:

`X_screen → F(X_screen) → MISDA → selected objective subset S`.

This screening stage does not use the analytical Pareto front and does not use either NSGA-III run.

For each MOP we then run:

`Full: X → F(X)`

`Reduced: X → F_S(X)`

The Reduced MOP below is only an objective projection adapter: it delegates all mathematical evaluation to the original MoeaBench MOP and slices the resulting objective matrix. No DTLZ or DPF formula is duplicated.


In [ ]:
class ObjectiveProjectionMOP(mb.mops.BaseMop):
    """Expose a subset of an existing MoeaBench MOP without duplicating it."""

    def __init__(self, source_mop, objective_indices):
        self.source_mop = source_mop
        self.objective_indices = tuple(int(i) for i in objective_indices)
        if len(self.objective_indices) < 2:
            raise ValueError("NSGA-III requires at least two selected objectives.")
        super().__init__(
            name=f"{source_mop.name}[MISDA]",
            M=len(self.objective_indices),
            N=source_mop.N,
            xl=np.asarray(source_mop.xl, dtype=float),
            xu=np.asarray(source_mop.xu, dtype=float),
        )

    def evaluation(self, X, n_ieq_constr=0):
        result = dict(self.source_mop.evaluation(X, n_ieq_constr))
        result["F"] = np.asarray(result["F"], dtype=float)[:, self.objective_indices]
        return result

    def ps(self, n_points=100):
        # Decision-space truth is unchanged; only the exposed objectives differ.
        return self.source_mop.ps(n_points)


In [ ]:
def _nd_front(F):
    """Return the non-dominated subset using MoeaBench's population algebra."""
    return np.asarray(Population(np.asarray(F, dtype=float)).non_dominated().objectives)


def _screen_misda(mop, *, n=MISDA_SAMPLE, seed=MISDA_SEED):
    rng = np.random.default_rng(seed)
    X = rng.uniform(
        np.asarray(mop.xl, dtype=float),
        np.asarray(mop.xu, dtype=float),
        size=(n, mop.N),
    )
    F = np.asarray(mop.evaluation(X)["F"], dtype=float)
    frame = pd.DataFrame(F, columns=[f"f{i + 1}" for i in range(mop.M)])
    mis_set = misda.discover(frame, name=f"{mop.name} screening", seed=seed)
    ranking = misda.rank(mis_set)
    selected = ranking.mis()
    return {
        "X": X,
        "F": frame,
        "mis_set": mis_set,
        "ranking": ranking,
        "selected": selected,
        "indices": tuple(int(i) for i in selected.indices),
    }


def _original_space_history(exp, original_mop):
    """Re-evaluate every generation's decision vectors on the original MOP."""
    return [
        _nd_front(original_mop.evaluation(np.asarray(X, dtype=float))["F"])
        for X in exp[0].history("x")
    ]


def _metric_value(metric, front, gt):
    return float(metric(np.asarray(front), ref=np.asarray(gt), progress=False))


def _igdplus_history(fronts, gt, label):
    values = [
        _metric_value(mb.metrics.igdplus, front, gt)
        for front in fronts
    ]
    return mb.metrics.MetricMatrix(
        np.asarray(values, dtype=float)[:, None],
        metric_name="IGD+",
        source_name=label,
    )


def _final_metrics(full_front, reduced_front, gt):
    gd_full = _metric_value(mb.metrics.gdplus, full_front, gt)
    gd_reduced = _metric_value(mb.metrics.gdplus, reduced_front, gt)
    igd_full = _metric_value(mb.metrics.igdplus, full_front, gt)
    igd_reduced = _metric_value(mb.metrics.igdplus, reduced_front, gt)

    hv_kwargs = dict(
        ref=np.asarray(gt),
        mode="auto",
        scale="abs",
        n_samples=HV_MC_SAMPLES,
        mc_seed=MOEA_SEED,
        progress=False,
    )
    hv_full = float(mb.metrics.hypervolume(np.asarray(full_front), **hv_kwargs))
    hv_reduced = float(mb.metrics.hypervolume(np.asarray(reduced_front), **hv_kwargs))

    return pd.DataFrame(
        {
            "Full": [gd_full, igd_full, hv_full],
            "Reduced": [gd_reduced, igd_reduced, hv_reduced],
            "Reduced - Full": [
                gd_reduced - gd_full,
                igd_reduced - igd_full,
                hv_reduced - hv_full,
            ],
        },
        index=["GD+", "IGD+", "HV"],
    )


In [ ]:
optimization_results = {}


def run_optimization_case(name, mop):
    screening = _screen_misda(mop)
    selected_indices = screening["indices"]
    selected_labels = [f"f{i + 1}" for i in selected_indices]

    reduced_mop = ObjectiveProjectionMOP(mop, selected_indices)

    full = mb.experiment(
        mop=mop,
        moea=mb.moeas.NSGA3(
            population=POPULATION,
            generations=GENERATIONS,
            seed=MOEA_SEED,
        ),
    )
    full.name = f"{name} — Full"

    reduced = mb.experiment(
        mop=reduced_mop,
        moea=mb.moeas.NSGA3(
            population=POPULATION,
            generations=GENERATIONS,
            seed=MOEA_SEED,
        ),
    )
    reduced.name = f"{name} — Reduced"

    full.run(repeat=1, silent=True)
    reduced.run(repeat=1, silent=True)

    # Separate NSGA-III reference-direction RNGs must not disturb paired search RNG.
    np.testing.assert_allclose(
        np.asarray(full[0].history("x")[0]),
        np.asarray(reduced[0].history("x")[0]),
    )

    # Ground truth always belongs to the original M-objective problem.
    gt_exp = mb.experiment(mop=mop)
    gt = np.asarray(gt_exp.optimal(n_points=GT_POINTS).objectives)

    full_history = _original_space_history(full, mop)
    reduced_history = _original_space_history(reduced, mop)
    full_front = full_history[-1]
    reduced_front = reduced_history[-1]

    metrics = _final_metrics(full_front, reduced_front, gt)

    diag_full = mb.clinic.audit(
        full_front,
        ground_truth=gt,
        initial_data=full_history[0],
        problem=name,
        k=POPULATION,
    )
    diag_full.experiment_name = "Full"

    diag_reduced = mb.clinic.audit(
        reduced_front,
        ground_truth=gt,
        initial_data=reduced_history[0],
        problem=name,
        k=POPULATION,
    )
    diag_reduced.experiment_name = "Reduced"

    igd_full = _igdplus_history(full_history, gt, "Full")
    igd_reduced = _igdplus_history(reduced_history, gt, "Reduced")

    result = {
        "name": name,
        "mop": mop,
        "screening": screening,
        "selected_indices": selected_indices,
        "selected_labels": selected_labels,
        "reduced_mop": reduced_mop,
        "full": full,
        "reduced": reduced,
        "gt": gt,
        "full_history": full_history,
        "reduced_history": reduced_history,
        "full_front": full_front,
        "reduced_front": reduced_front,
        "metrics": metrics,
        "diag_full": diag_full,
        "diag_reduced": diag_reduced,
        "igd_full": igd_full,
        "igd_reduced": igd_reduced,
    }
    optimization_results[name] = result

    print(f"{name}: MISDA reduction {mop.M} → {len(selected_indices)}")
    print(f"Selected objectives: {', '.join(selected_labels)}")
    print(
        f"Budget per treatment: population={POPULATION}, "
        f"generations={GENERATIONS}, evaluations≈{POPULATION * GENERATIONS}"
    )
    display(metrics)

    mb.view.topology(
        full_front,
        reduced_front,
        gt=gt,
        show_gt=True,
        objectives=[0, 1, 2],
        labels=["Full", "Reduced"],
        title=f"{name}: Full vs Reduced in original objective space (f1–f3 projection)",
    )

    mb.view.radar(
        diag_full,
        diag_reduced,
        title=f"{name}: clinical quality in original objective space",
    )

    mb.view.history(
        igd_full,
        igd_reduced,
        title=f"{name}: IGD+ convergence in original objective space",
    )

    return result


## Benchmark battery

All problems are taken directly from the pinned MoeaBench dependency.

- **DTLZ2** — regular smooth, non-degenerate control.
- **DTLZ5** — classical degenerate front.
- **DTLZ7** — disconnected Pareto front.
- **DPF1** — simple degenerate projection.
- **DPF3** — nonlinear min/max chaotic projection.
- **DPF5** — conditional objective construction.

DPF problems use intrinsic base dimension `D=2`. DPF5 requires an explicit decision dimension large enough to contain its `x_M` term.


In [ ]:
PROBLEMS = {
    "DTLZ2": mb.mops.DTLZ2(M=M),
    "DTLZ5": mb.mops.DTLZ5(M=M),
    "DTLZ7": mb.mops.DTLZ7(M=M),
    "DPF1": mb.mops.DPF1(M=M, D=2, K=5),
    "DPF3": mb.mops.DPF3(M=M, D=2, K=5),
    "DPF5": mb.mops.DPF5(M=M, D=2, K=5, N=M + 5 - 1),
}

[(name, mop.M, mop.N) for name, mop in PROBLEMS.items()]


## DTLZ2 — regular smooth control

The non-degenerate control checks that MISDA does not create an artificial optimization advantage when little or no objective reduction is justified.


In [ ]:
dtlz2 = run_optimization_case("DTLZ2", PROBLEMS["DTLZ2"])


## DTLZ5 — classical degeneracy

DTLZ5 tests whether a structurally lower-dimensional Pareto geometry can be reduced without losing original-space optimization quality.


In [ ]:
dtlz5 = run_optimization_case("DTLZ5", PROBLEMS["DTLZ5"])


## DTLZ7 — disconnected front

DTLZ7 tests whether objective reduction preserves disconnected trade-off regions rather than merely a smooth manifold.


In [ ]:
dtlz7 = run_optimization_case("DTLZ7", PROBLEMS["DTLZ7"])


## DPF1 — linear degenerate projection

DPF1 is a positive control for high-dimensional objectives generated from a low-dimensional base front.


In [ ]:
dpf1 = run_optimization_case("DPF1", PROBLEMS["DPF1"])


## DPF3 — nonlinear degenerate projection

DPF3 adds nonlinear min/max projection structure and tests whether a reduced objective set remains an adequate optimization target.


In [ ]:
dpf3 = run_optimization_case("DPF3", PROBLEMS["DPF3"])


## DPF5 — conditional structure

DPF5 changes objective construction according to the decision-space regime and therefore tests reduction under conditional dependence.


In [ ]:
dpf5 = run_optimization_case("DPF5", PROBLEMS["DPF5"])


# Suite summary

The summary keeps the scientific axes separate: reduction obtained, final original-space quality, and convergence. A lower GD+/IGD+ and a higher HV indicate better final approximation, but the table deliberately reports values and deltas rather than declaring a winner.


In [ ]:
summary_rows = []
for name, result in optimization_results.items():
    metrics = result["metrics"]
    summary_rows.append(
        {
            "Problem": name,
            "Objectives": result["mop"].M,
            "Selected": len(result["selected_indices"]),
            "GD+ Full": metrics.loc["GD+", "Full"],
            "GD+ Reduced": metrics.loc["GD+", "Reduced"],
            "IGD+ Full": metrics.loc["IGD+", "Full"],
            "IGD+ Reduced": metrics.loc["IGD+", "Reduced"],
            "HV Full": metrics.loc["HV", "Full"],
            "HV Reduced": metrics.loc["HV", "Reduced"],
            "Final IGD+ Δ": metrics.loc["IGD+", "Reduced - Full"],
        }
    )

optimization_summary = pd.DataFrame(summary_rows)
optimization_summary
